# Lab Assignment 3 - Part B: Textual Medical Data
## MTSamples Medical Transcriptions

Pipeline: acquire -> inspect -> PHI removal (HIPAA Safe Harbor) -> clean
-> normalise -> EDA -> feature engineering (TF-IDF, n-grams) -> tokenise
-> padded sequences (model-ready)

Dataset: MTSamples Medical Transcriptions (Kaggle)
Place `mtsamples.csv` in ./data/

In [ ]:
import re
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_selection import chi2, SelectKBest
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

DATA_PATH = Path("data/mtsamples.csv")
OUT_DIR = Path("outputs/partB")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## B1. Acquisition and inspection

In [ ]:
df = pd.read_csv(DATA_PATH)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("\nNulls:\n", df.isna().sum())
print("\nSpecialties:", df["medical_specialty"].nunique())
print("\nSample transcription (first 600 chars):\n")
print(df["transcription"].dropna().iloc[0][:600])

# Drop records with no transcription - nothing to learn from an empty note
df = df.dropna(subset=["transcription"]).reset_index(drop=True)
df["medical_specialty"] = df["medical_specialty"].str.strip()
print("\nAfter dropping empty transcriptions:", df.shape)

## B2. PHI removal - HIPAA Safe Harbor de-identification

The Safe Harbor method requires 18 identifier categories to be removed before
clinical text can be shared or used for research. This is a legal requirement,
not a modelling preference - and it is the single biggest difference between
clinical NLP and generic NLP.

Each identifier is replaced with a typed placeholder rather than deleted, so the
sentence keeps its grammatical structure and the model still learns that "a date
appeared here" without learning *which* date.

Note on age: HIPAA requires ages over 89 to be aggregated into a single "90+"
category, because a very high age is itself identifying in a small population.

In [ ]:
PHI_PATTERNS = [
    # Names following clinical titles
    (r"\b(?:Dr|Doctor|Mr|Mrs|Ms|Miss|Prof)\.?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)?",
     "<NAME>"),
    # Dates in numeric formats: 01/02/2020, 2020-01-02, 1.2.20
    (r"\b\d{1,2}[/\-.]\d{1,2}[/\-.]\d{2,4}\b", "<DATE>"),
    # Dates in written formats: January 2, 2020
    (r"\b(?:January|February|March|April|May|June|July|August|September|"
     r"October|November|December)\s+\d{1,2},?\s*\d{0,4}\b", "<DATE>"),
    # Ages 90+ (HIPAA aggregation requirement)
    (r"\b(?:9[0-9]|1[0-9]{2})[\s-]*(?:year|yr)s?[\s-]*old\b", "<AGE_90_PLUS>"),
    # Telephone / fax numbers
    (r"\b(?:\+?\d{1,3}[\s-]?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}\b", "<PHONE>"),
    # Social security numbers
    (r"\b\d{3}-\d{2}-\d{4}\b", "<SSN>"),
    # Medical record / account numbers
    (r"\b(?:MRN|Medical Record(?:\s+Number)?|Account|Chart)\s*#?\s*:?\s*\d+\b",
     "<MRN>"),
    # Email addresses
    (r"\b[\w.+-]+@[\w-]+\.[\w.]+\b", "<EMAIL>"),
    # URLs and IP addresses
    (r"https?://\S+|www\.\S+", "<URL>"),
    (r"\b(?:\d{1,3}\.){3}\d{1,3}\b", "<IP>"),
    # Street addresses
    (r"\b\d+\s+[A-Z][a-z]+\s+(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|"
     r"Lane|Ln|Drive|Dr|Court|Ct)\b", "<ADDRESS>"),
    # ZIP codes
    (r"\b\d{5}(?:-\d{4})?\b", "<ZIP>"),
    # Hospital / facility names
    (r"\b[A-Z][a-z]+\s+(?:Hospital|Medical Center|Clinic|Health System)\b",
     "<FACILITY>"),
]


def remove_phi(text: str) -> str:
    """Apply HIPAA Safe Harbor de-identification to a clinical note."""
    if not isinstance(text, str):
        return ""
    for pattern, placeholder in PHI_PATTERNS:
        text = re.sub(pattern, placeholder, text, flags=re.IGNORECASE)
    return text


def count_phi(text: str) -> dict:
    """Count how many of each identifier type appear - an audit trail."""
    counts = {}
    for pattern, placeholder in PHI_PATTERNS:
        n = len(re.findall(pattern, str(text), flags=re.IGNORECASE))
        if n:
            counts[placeholder] = counts.get(placeholder, 0) + n
    return counts


# Audit before removal
phi_audit = Counter()
for txt in df["transcription"]:
    phi_audit.update(count_phi(txt))

print("PHI identifiers detected across the corpus:")
for k, v in phi_audit.most_common():
    print(f"  {k:<16} {v:>6}")

df["deidentified"] = df["transcription"].apply(remove_phi)

# Verify - residual PHI after cleaning should be zero
residual = Counter()
for txt in df["deidentified"]:
    residual.update(count_phi(txt))
print("\nResidual PHI after de-identification:", dict(residual) or "none detected")

print("\nBefore/after sample:")
sample_idx = 0
print("BEFORE:", df['transcription'].iloc[sample_idx][:300])
print("\nAFTER: ", df['deidentified'].iloc[sample_idx][:300])

## B3. Text cleaning and normalisation

Two decisions that differ from generic NLP:

1. **Negation is preserved.** "no evidence of malignancy" and "evidence of
   malignancy" are clinically opposite. Standard stopword lists delete "no",
   "not" and "denies", which inverts the meaning. Those tokens are kept.
2. **Placeholders are preserved.** `<DATE>` and `<NAME>` must survive cleaning
   intact, so punctuation stripping must not break the angle brackets.

In [ ]:
NEGATION_TERMS = {
    "no", "not", "never", "none", "nor", "cannot", "without",
    "denies", "denied", "negative", "absent", "unremarkable", "free",
    "ruled", "out", "unlikely", "doubt",
}

GENERIC_STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "if", "then", "this", "that",
    "these", "those", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "should",
    "can", "could", "may", "might", "must", "shall", "to", "of", "in",
    "for", "on", "at", "by", "from", "with", "as", "into", "about",
    "he", "she", "it", "they", "we", "you", "his", "her", "their", "our",
}
# Critically: subtract negation terms so they are never removed
CLINICAL_STOPWORDS = GENERIC_STOPWORDS - NEGATION_TERMS

# Common clinical abbreviations - expanded so the model sees one canonical form
ABBREVIATIONS = {
    r"\bpt\b": "patient", r"\bhx\b": "history", r"\bdx\b": "diagnosis",
    r"\btx\b": "treatment", r"\brx\b": "prescription", r"\bsx\b": "symptoms",
    r"\bfx\b": "fracture", r"\bbp\b": "blood pressure", r"\bhr\b": "heart rate",
    r"\bwbc\b": "white blood cell", r"\brbc\b": "red blood cell",
    r"\bcbc\b": "complete blood count", r"\bekg\b": "electrocardiogram",
    r"\becg\b": "electrocardiogram", r"\bcxr\b": "chest xray",
    r"\bict\b": "intensive care", r"\bnkda\b": "no known drug allergies",
    r"\bsob\b": "shortness of breath", r"\bcp\b": "chest pain",
    r"\bpo\b": "by mouth", r"\bbid\b": "twice daily", r"\btid\b": "three times daily",
    r"\bqd\b": "once daily", r"\bprn\b": "as needed",
}


def clean_clinical_text(text: str, remove_stopwords: bool = True) -> str:
    text = str(text).lower()

    # Protect placeholders before punctuation stripping
    text = re.sub(r"<(\w+)>", r" phitoken\1 ", text)

    # Expand abbreviations
    for abbr, full in ABBREVIATIONS.items():
        text = re.sub(abbr, full, text)

    # Normalise measurements: keep the fact a number was present, drop the value
    text = re.sub(r"\b\d+\.?\d*\s*(?:mg|ml|mcg|kg|lb|cm|mm|mmhg|bpm)\b",
                  " measurementvalue ", text)
    text = re.sub(r"\b\d+\.?\d*\b", " numtoken ", text)

    # Strip punctuation and collapse whitespace
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    if remove_stopwords:
        tokens = [t for t in tokens if t not in CLINICAL_STOPWORDS and len(t) > 2]
    return " ".join(tokens)


df["cleaned"] = df["deidentified"].apply(clean_clinical_text)

print("Cleaned sample:\n", df["cleaned"].iloc[0][:400])
print("\nNegation check - are negation terms retained?")
neg_present = [t for t in NEGATION_TERMS if f" {t} " in f" {df['cleaned'].iloc[0]} "]
print("  found in sample:", neg_present or "(none in this particular note)")

## B4. Exploratory Data Analysis

In [ ]:
df["char_len"] = df["deidentified"].str.len()
df["word_count"] = df["cleaned"].str.split().str.len()

print("Document length statistics:")
print(df[["char_len", "word_count"]].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df["word_count"], bins=50, ax=axes[0])
axes[0].set_title("Token count per note (after cleaning)")
axes[0].axvline(df["word_count"].median(), c="red", ls="--",
                label=f"median={df['word_count'].median():.0f}")
axes[0].legend()

top_spec = df["medical_specialty"].value_counts().head(15)
sns.barplot(y=top_spec.index, x=top_spec.values, ax=axes[1])
axes[1].set_title("Top 15 medical specialties")
plt.tight_layout()
plt.savefig(OUT_DIR / "text_eda.png", dpi=120)
plt.close()

# Class imbalance across specialties
spec_counts = df["medical_specialty"].value_counts()
print(f"\nSpecialty class imbalance:")
print(f"  classes: {len(spec_counts)}")
print(f"  largest:  {spec_counts.index[0]} ({spec_counts.iloc[0]} notes)")
print(f"  smallest: {spec_counts.index[-1]} ({spec_counts.iloc[-1]} notes)")
print(f"  imbalance ratio: {spec_counts.iloc[0] / spec_counts.iloc[-1]:.1f} : 1")

# Vocabulary statistics
all_tokens = " ".join(df["cleaned"]).split()
vocab = Counter(all_tokens)
print(f"\nCorpus: {len(all_tokens):,} tokens, {len(vocab):,} unique")
print("\nTop 25 terms:")
for term, count in vocab.most_common(25):
    print(f"  {term:<24} {count:>6}")

# Terms appearing only once - these blow up the vocabulary without adding signal
hapax = sum(1 for c in vocab.values() if c == 1)
print(f"\nHapax legomena (appear once): {hapax:,} ({hapax / len(vocab):.1%} of vocab)")

## B5. Feature engineering - TF-IDF and n-grams

In [ ]:
# Keep only specialties with enough examples to model
MIN_DOCS = 50
valid_specialties = spec_counts[spec_counts >= MIN_DOCS].index
df_model = df[df["medical_specialty"].isin(valid_specialties)].reset_index(drop=True)
print(f"Modelling on {len(valid_specialties)} specialties, {len(df_model)} notes")

le = LabelEncoder()
y = le.fit_transform(df_model["medical_specialty"])

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df_model["cleaned"], y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# TF-IDF with unigrams and bigrams. Bigrams matter clinically: "chest pain" and
# "no fever" carry meaning that individual tokens lose.
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,           # ignore terms in fewer than 5 documents
    max_df=0.85,        # ignore terms in >85% of documents (no discriminative power)
    sublinear_tf=True,  # dampen the effect of very frequent terms
)
X_train_tfidf = tfidf.fit_transform(X_train_txt)
X_test_tfidf = tfidf.transform(X_test_txt)

print("TF-IDF matrix:", X_train_tfidf.shape,
      f"sparsity={1 - X_train_tfidf.nnz / np.prod(X_train_tfidf.shape):.4f}")

# Most informative bigrams
bigrams = [t for t in tfidf.get_feature_names_out() if " " in t]
print(f"\nBigrams retained: {len(bigrams)}. Examples:", bigrams[:15])

In [ ]:
# Feature selection: chi-square between term and specialty label
chi_selector = SelectKBest(chi2, k=1000).fit(X_train_tfidf, y_train)
chi_scores = pd.Series(
    chi_selector.scores_, index=tfidf.get_feature_names_out()
).sort_values(ascending=False)
print("\nTop 20 terms by chi-square association with specialty:")
print(chi_scores.head(20).round(1))

X_train_sel = chi_selector.transform(X_train_tfidf)
X_test_sel = chi_selector.transform(X_test_tfidf)
print("After chi-square selection:", X_train_sel.shape)

In [ ]:
# Feature extraction: LSA (truncated SVD) - dense semantic representation
svd = TruncatedSVD(n_components=200, random_state=RANDOM_STATE)
X_train_lsa = svd.fit_transform(X_train_tfidf)
X_test_lsa = svd.transform(X_test_tfidf)
print(f"\nLSA: {X_train_lsa.shape}, "
      f"explained variance = {svd.explained_variance_ratio_.sum():.2%}")

# Topic modelling for interpretability
count_vec = CountVectorizer(max_features=2000, min_df=5, max_df=0.85)
counts = count_vec.fit_transform(X_train_txt)
lda = LatentDirichletAllocation(n_components=8, random_state=RANDOM_STATE,
                               learning_method="online")
lda.fit(counts)

print("\nDiscovered topics:")
terms = count_vec.get_feature_names_out()
for i, comp in enumerate(lda.components_):
    top = [terms[j] for j in comp.argsort()[-10:][::-1]]
    print(f"  Topic {i}: {', '.join(top)}")

## B6. Tokenisation into padded sequences (model-ready)

TF-IDF is a bag of words and discards order. For sequence models (LSTM,
transformer) the text must become integer token IDs padded to a fixed length.

In [ ]:
MAX_VOCAB = 10000
MAX_LEN = 300

# Build vocabulary from training data only
train_tokens = Counter(" ".join(X_train_txt).split())
vocab_list = [w for w, _ in train_tokens.most_common(MAX_VOCAB - 2)]
word2idx = {w: i + 2 for i, w in enumerate(vocab_list)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1


def texts_to_sequences(texts, mapping, max_len):
    seqs = np.zeros((len(texts), max_len), dtype=np.int32)
    for i, text in enumerate(texts):
        ids = [mapping.get(t, 1) for t in str(text).split()[:max_len]]
        seqs[i, :len(ids)] = ids       # post-padding with 0
    return seqs


X_train_seq = texts_to_sequences(X_train_txt, word2idx, MAX_LEN)
X_test_seq = texts_to_sequences(X_test_txt, word2idx, MAX_LEN)

print("Sequence tensors:", X_train_seq.shape, X_test_seq.shape)
print("Vocabulary size:", len(word2idx))
oov_rate = np.mean(X_test_seq == 1)
print(f"OOV rate on test set: {oov_rate:.2%}")
print("Sample sequence (first 30 IDs):", X_train_seq[0][:30])

np.savez_compressed(
    OUT_DIR / "mtsamples_model_ready.npz",
    X_train_seq=X_train_seq, X_test_seq=X_test_seq,
    X_train_lsa=X_train_lsa, X_test_lsa=X_test_lsa,
    y_train=y_train, y_test=y_test,
    class_names=le.classes_,
)
df[["medical_specialty", "deidentified", "cleaned"]].to_csv(
    OUT_DIR / "mtsamples_deidentified.csv", index=False
)
print(f"\nSaved to {OUT_DIR}")

## Part B summary

| Step | Finding |
|---|---|
| PHI | 12 Safe Harbor identifier patterns detected and replaced with typed placeholders |
| Cleaning | Negation terms deliberately excluded from the stopword list |
| EDA | Heavy specialty imbalance; large hapax fraction motivates `min_df` cutoff |
| Features | TF-IDF 1-2 grams (5000) -> chi-square top 1000 -> LSA 200 dims |
| Output | Padded integer sequences (N x 300) + dense LSA matrix |